# Integración: reconocimiento y generación de voz

Notebook propio para la Actividad 7 de la práctica. El flujo conecta detección de voz, transcripción, generación con Qwen3-TTS y comparación de hablante con ECAPA. Permite grabar desde el micrófono o cargar WAV locales de `data/`.

**Uso responsable:** utiliza una grabación propia o una voz cuyo uso tengas autorizado. Identifica los audios generados como sintéticos y no compartas grabaciones privadas.

## Preparar el entorno

En Anaconda Prompt crea/activa el entorno siguiendo la configuración de la práctica. En Windows con GPU NVIDIA compatible con CUDA 12.8:

```bash
conda create -n UNIE_VozClon python=3.12 pip
conda activate UNIE_VozClon
python -m pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
python -m pip install qwen-tts==0.1.1 ipykernel
python -m pip install matplotlib speechbrain==1.0.3 pandas soundfile==0.13.1 sounddevice
python -m ipykernel install --user --name UNIE_VozClon --display-name "Python 3.12 (UNIE_VozClon)"
```

Sin GPU NVIDIA compatible, sustituye solo la instalación de PyTorch/TorchAudio por `--index-url https://download.pytorch.org/whl/cpu`. Selecciona el kernel **Python 3.12 (UNIE_VozClon)** y reinícialo tras instalar; no uses el kernel Python 3.8 del proyecto de cámara. Para combinar Qwen3-TTS con SpeechBrain no fijes `huggingface-hub==0.30.2`: Transformers de Qwen necesita una versión más reciente y `pip` debe resolverla. Los pesos se descargan la primera vez que se cargan los modelos.

Por defecto, las celdas siguientes graban dos audios desde el micrófono: una referencia y una segunda toma real de la misma persona. También puedes usar WAV ya grabados: cambia `RECORD_FROM_MIC` a `False` y coloca los archivos en las rutas indicadas. Para grabar, instala `sounddevice` en el kernel del notebook y selecciona una entrada válida.

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchaudio
from IPython.display import Audio, display

DATA_DIR = Path('data')
REFERENCE_PATH = DATA_DIR / 'referencia.wav'
SECOND_SPEAKER_PATH = DATA_DIR / 'segunda_grabacion.wav'
OUTPUT_DIR = DATA_DIR / 'generados'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECORD_FROM_MIC = True
RECORD_SECONDS = 6
MIC_DEVICE = None  # None usa el micrófono predeterminado; también puedes indicar su número.

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
print(f'Archivo de referencia: {REFERENCE_PATH.resolve()}')
print(f'Segunda grabación: {SECOND_SPEAKER_PATH.resolve()}')

## Grabar desde el micrófono (opcional)

Esta celda sigue el flujo de grabación del notebook del profesor. Si se usa el micrófono, se guardan dos WAV nuevos con nombres únicos para no sobrescribir capturas anteriores. Pulsa Enter cuando estés listo para cada toma. Si prefieres usar archivos existentes, cambia `RECORD_FROM_MIC = False` en la celda anterior.

In [ ]:
if RECORD_FROM_MIC:
    try:
        import sounddevice as sd
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            'Selecciona primero Python 3.12 (UNIE_VozClon) e instala sounddevice allí: '
            '`python -m pip install sounddevice`, y después reinícialo.'
        ) from exc

    host_api = next(
        (api for api in sd.query_hostapis() if api['name'] == 'Windows WASAPI'),
        None,
    )
    input_devices = [
        index for index, info in enumerate(sd.query_devices())
        if info['max_input_channels'] > 0
    ]
    if not input_devices:
        raise RuntimeError('No se encontraron micrófonos de entrada disponibles.')
    for index in input_devices:
        print(index, sd.query_devices(index)['name'])

    if MIC_DEVICE is None:
        default_device = host_api['default_input_device'] if host_api else sd.default.device[0]
        MIC_DEVICE = default_device if default_device in input_devices else None
    if MIC_DEVICE not in input_devices:
        raise RuntimeError('Elige en MIC_DEVICE el número de un micrófono de la lista.')

    MIC_SAMPLE_RATE = int(sd.query_devices(MIC_DEVICE, 'input')['default_samplerate'])
    sd.check_input_settings(
        device=MIC_DEVICE,
        channels=1,
        dtype='float32',
        samplerate=MIC_SAMPLE_RATE,
    )

    def record_clip(filename_prefix, prompt):
        input(f'{prompt} Pulsa Enter para empezar a grabar {RECORD_SECONDS} segundos...')
        recording = None
        status = None
        try:
            recording = sd.rec(
                round(RECORD_SECONDS * MIC_SAMPLE_RATE),
                samplerate=MIC_SAMPLE_RATE,
                channels=1,
                dtype='float32',
                device=MIC_DEVICE,
            )
            status = sd.wait()
        finally:
            sd.stop()
        if status:
            raise RuntimeError(f'La captura quedó incompleta: {status}. Vuelve a grabar.')
        samples = torch.from_numpy(recording[:, 0].copy())
        if not torch.isfinite(samples).all() or torch.max(torch.abs(samples)).item() == 0:
            raise ValueError('La grabación está vacía o contiene solo ceros; vuelve a grabar.')
        from datetime import datetime
        output_path = DATA_DIR / f'{filename_prefix}_{datetime.now():%Y-%m-%d_%H-%M-%S_%f}.wav'
        torchaudio.save(str(output_path), samples.unsqueeze(0), MIC_SAMPLE_RATE)
        print(f'Grabación guardada: {output_path}')
        display(Audio(samples.numpy(), rate=MIC_SAMPLE_RATE))
        return output_path

    REFERENCE_PATH = record_clip(
        'referencia',
        'Di una frase completa para usarla como referencia de voz.',
    )
    SECOND_SPEAKER_PATH = record_clip(
        'segunda_grabacion',
        'Vuelve a hablar con la misma voz; puedes decir otra frase.',
    )
else:
    print(f'Usando WAV existentes: {REFERENCE_PATH} y {SECOND_SPEAKER_PATH}')

## 1. Cargar y explorar el audio

La señal original se conserva con su frecuencia de muestreo real. Para analizar canales múltiples, se crea una copia mono promediando los canales.

In [ ]:
def load_audio(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'No existe el archivo WAV: {path.resolve()}')
    waveform, sample_rate = torchaudio.load(str(path))
    if waveform.numel() == 0 or sample_rate <= 0:
        raise ValueError(f'Audio vacío o frecuencia no válida: {path}')
    if not torch.isfinite(waveform).all():
        raise ValueError(f'El audio contiene muestras no válidas: {path}')
    if torch.max(torch.abs(waveform)).item() == 0:
        raise ValueError(f'El audio contiene solo ceros: {path}')
    return waveform, sample_rate


def to_mono(waveform):
    if waveform.ndim != 2 or waveform.shape[0] < 1:
        raise ValueError('Se esperaba un audio con forma canales x muestras.')
    return waveform.mean(dim=0)


reference_audio, reference_sr = load_audio(REFERENCE_PATH)
reference_mono = to_mono(reference_audio)
duration_s = reference_audio.shape[1] / reference_sr
print(f'Frecuencia: {reference_sr} Hz | Canales: {reference_audio.shape[0]} | '
      f'Muestras: {reference_audio.shape[1]} | Duración: {duration_s:.2f} s')

time_s = torch.arange(reference_mono.numel()) / reference_sr
plt.figure(figsize=(14, 3))
plt.plot(time_s.numpy(), reference_mono.numpy(), linewidth=0.5)
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.title('Audio de referencia (mono para análisis)')
plt.grid(True)
waveform_plot_path = DATA_DIR / 'forma_onda_referencia.png'
plt.savefig(waveform_plot_path, dpi=150, bbox_inches='tight')
print(f'Gráfica guardada en: {waveform_plot_path.resolve()}')
plt.show()

## 2. Detectar intervalos con habla

Este VAD sencillo usa RMS por ventanas: conserva ventanas por encima de un umbral absoluto y relativo al máximo, une pausas breves y descarta intervalos muy cortos. Detectar energía no garantiza detectar habla; revisa el gráfico y ajusta los parámetros si hay ruido o silencios.

In [ ]:
WINDOW_MS = 30
RELATIVE_THRESHOLD = 0.12
ABSOLUTE_THRESHOLD = 0.008
MIN_SPEECH_S = 0.25
MERGE_SILENCE_S = 0.35


def detect_speech_intervals(audio, sample_rate):
    window = max(1, round(sample_rate * WINDOW_MS / 1000))
    frame_count = math.ceil(audio.numel() / window)
    padded = torch.nn.functional.pad(audio, (0, frame_count * window - audio.numel()))
    frames = padded.reshape(frame_count, window)
    rms = torch.sqrt(torch.mean(frames.square(), dim=1) + 1e-12)
    peak = float(rms.max()) if rms.numel() else 0.0
    threshold = max(ABSOLUTE_THRESHOLD, peak * RELATIVE_THRESHOLD)
    active = (rms >= threshold).tolist()

    spans = []
    start = None
    for index, is_active in enumerate(active + [False]):
        if is_active and start is None:
            start = index
        elif not is_active and start is not None:
            spans.append([start * window / sample_rate,
                          min(index * window / sample_rate, audio.numel() / sample_rate)])
            start = None

    merged = []
    for begin, end in spans:
        if merged and begin - merged[-1][1] <= MERGE_SILENCE_S:
            merged[-1][1] = end
        else:
            merged.append([begin, end])
    return [(begin, end) for begin, end in merged
            if end - begin >= MIN_SPEECH_S], rms, threshold


speech_intervals, frame_rms, vad_threshold = detect_speech_intervals(reference_mono, reference_sr)
if not speech_intervals:
    raise RuntimeError('No se detectó habla. Comprueba el WAV o ajusta los umbrales del VAD.')

print(f'Umbral RMS: {vad_threshold:.4f}')
for index, (begin, end) in enumerate(speech_intervals):
    print(f'{index}: {begin:.2f}–{end:.2f} s ({end - begin:.2f} s)')

frame_times = (torch.arange(frame_rms.numel()) * WINDOW_MS / 1000).numpy()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(time_s.numpy(), reference_mono.numpy(), linewidth=0.5, label='Audio')
ax2 = ax.twinx()
ax2.plot(frame_times, frame_rms.numpy(), color='tab:orange', alpha=0.75, label='RMS')
ax2.axhline(vad_threshold, color='tab:red', linestyle='--', label='Umbral VAD')
for begin, end in speech_intervals:
    ax.axvspan(begin, end, color='tab:green', alpha=0.2)
ax.set(xlabel='Tiempo (s)', ylabel='Amplitud', title='Intervalos detectados (verde)')
ax2.set_ylabel('RMS')
ax.grid(True)
plt.show()

Escoge un intervalo del gráfico que contenga una frase completa. Los márgenes permiten incluir un poco de contexto; revísalos para no cortar palabras.

In [ ]:
INTERVAL_INDEX = 0
START_MARGIN_S = 0.10
END_MARGIN_S = 0.10

if not 0 <= INTERVAL_INDEX < len(speech_intervals):
    raise IndexError('INTERVAL_INDEX no corresponde a ningún intervalo detectado.')
interval_start, interval_end = speech_intervals[INTERVAL_INDEX]
clip_start_s = max(0.0, interval_start - START_MARGIN_S)
clip_end_s = min(duration_s, interval_end + END_MARGIN_S)
start_sample = round(clip_start_s * reference_sr)
end_sample = round(clip_end_s * reference_sr)
reference_clip = reference_mono[start_sample:end_sample].contiguous()
if reference_clip.numel() == 0:
    raise ValueError('El fragmento seleccionado está vacío; revisa el intervalo y los márgenes.')

print(f'Fragmento: {clip_start_s:.2f}–{clip_end_s:.2f} s, {reference_sr} Hz')
display(Audio(reference_clip.numpy(), rate=reference_sr))

## 3. Transcribir con Wav2Vec2 / VoxPopuli

El bundle requiere 16 kHz. Aquí se remuestrean las muestras, no se cambia únicamente la etiqueta de frecuencia. La decodificación greedy elimina repeticiones consecutivas y el símbolo de separación.

In [ ]:
ASR_BUNDLE = torchaudio.pipelines.VOXPOPULI_ASR_BASE_10K_ES
asr_model = ASR_BUNDLE.get_model(
    dl_kwargs={'model_dir': str(DATA_DIR), 'weights_only': True}
).to(DEVICE).eval()
asr_model.requires_grad_(False)
asr_labels = ASR_BUNDLE.get_labels()
ASR_SAMPLE_RATE = ASR_BUNDLE.sample_rate

clip_16k = torchaudio.functional.resample(reference_clip, reference_sr, ASR_SAMPLE_RATE)
with torch.inference_mode():
    emissions, _ = asr_model(clip_16k.unsqueeze(0).to(DEVICE))
    best_path = emissions[0].argmax(dim=-1).cpu()

collapsed_path = torch.unique_consecutive(best_path)
automatic_transcript = ''.join(
    asr_labels[token] for token in collapsed_path.tolist() if token != 0
)
automatic_transcript = ' '.join(automatic_transcript.replace('|', ' ').split()).lower()
print('Transcripción automática:', automatic_transcript or '[vacía]')
if not automatic_transcript:
    raise RuntimeError('Wav2Vec2 no produjo una transcripción. Revisa el fragmento y vuelve a transcribir.')

Compara la transcripción automática con lo que escuchas y edita el texto revisado. Debe corresponder a las palabras del fragmento seleccionado, no a los mensajes nuevos que se generarán.

In [ ]:
REFERENCE_TRANSCRIPT = automatic_transcript  # Corrige aquí tras escuchar el fragmento.
if not REFERENCE_TRANSCRIPT.strip():
    raise ValueError('La transcripción revisada está vacía; no se puede crear el prompt de voz.')
print('Transcripción que se usará con la referencia:', REFERENCE_TRANSCRIPT)

## 4. Generar mensajes nuevos con Qwen3-TTS

Se carga una vez el modelo Base 0.6B y se reutiliza el prompt de voz. Los textos nuevos deben ser distintos de la transcripción de referencia. Se usa español; la salida es sintética.

In [ ]:
from qwen_tts import Qwen3TTSModel
import numpy as np

MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
MODEL_DTYPE = torch.bfloat16 if DEVICE.startswith('cuda') else torch.float32
tts_model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    dtype=MODEL_DTYPE,
)

voice_prompt = tts_model.create_voice_clone_prompt(
    ref_audio=(reference_clip.numpy().astype(np.float32), reference_sr),
    ref_text=REFERENCE_TRANSCRIPT,
    x_vector_only_mode=False,
)

NEW_TEXTS = [
    'Hoy hace un día tranquilo y agradable.',
    '¿A qué hora empieza la actividad de mañana?',
    'El paquete contiene doce libros y cuatro cuadernos.',
]

if len(NEW_TEXTS) not in (3, 4):
    raise ValueError('La práctica requiere tres o cuatro mensajes nuevos.')
if any(not text.strip() for text in NEW_TEXTS):
    raise ValueError('Los mensajes nuevos no pueden estar vacíos.')
if any(text.strip().casefold() == REFERENCE_TRANSCRIPT.strip().casefold() for text in NEW_TEXTS):
    raise ValueError('Cada mensaje nuevo debe ser distinto de la transcripción de referencia.')
print(f'Mensajes a generar: {len(NEW_TEXTS)}')

In [ ]:
generated = []
for index, text in enumerate(NEW_TEXTS, start=1):
    wavs, generated_sr = tts_model.generate_voice_clone(
        text=text,
        language='Spanish',
        voice_clone_prompt=voice_prompt,
    )
    if not wavs or generated_sr <= 0:
        raise RuntimeError(f'Qwen no devolvió audio válido para el mensaje {index}.')
    audio_array = np.asarray(wavs[0], dtype=np.float32).squeeze()
    if audio_array.ndim != 1 or audio_array.size == 0:
        raise ValueError(f'Forma de audio inesperada para el mensaje {index}: {audio_array.shape}')
    output_path = OUTPUT_DIR / f'mensaje_sintetico_{index:02d}.wav'
    torchaudio.save(str(output_path), torch.from_numpy(audio_array).unsqueeze(0), generated_sr)
    generated.append({
        'index': index,
        'text': text,
        'path': output_path,
        'audio': torch.from_numpy(audio_array),
        'sample_rate': generated_sr,
    })
    print(f'Guardado: {output_path} ({generated_sr} Hz)')
    display(Audio(audio_array, rate=generated_sr))

Escucha y completa la valoración manual. Evalúa por separado el contenido, la semejanza vocal, la naturalidad y los artefactos; una puntuación automática no reemplaza esta revisión.

In [ ]:
manual_review = pd.DataFrame([
    {
        'archivo': item['path'].name,
        'texto': item['text'],
        'contenido correcto (sí/no/parcial)': '',
        'semejanza vocal (1-5)': '',
        'naturalidad (1-5)': '',
        'artefactos / observaciones': '',
    }
    for item in generated
])
manual_review

## 5. Comparar hablantes con ECAPA

Se comparan cada audio generado, la referencia seleccionada y una segunda grabación real de la misma persona. ECAPA recibe copias mono a 16 kHz; los WAV originales y los generados se mantienen intactos.

In [ ]:
import torch.nn.functional as F
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

if not SECOND_SPEAKER_PATH.is_file():
    raise FileNotFoundError(
        f'Falta la segunda grabación real del mismo hablante: {SECOND_SPEAKER_PATH.resolve()}'
    )

ecapa = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir=str(DATA_DIR / 'modelos' / 'ecapa_tdnn'),
    run_opts={'device': DEVICE},
    local_strategy=LocalStrategy.COPY_SKIP_CACHE,
)
ecapa.eval()
ecapa.requires_grad_(False)
ECAPA_SAMPLE_RATE = 16000


def prepare_for_ecapa(audio, sample_rate):
    mono = to_mono(audio).float()
    if sample_rate != ECAPA_SAMPLE_RATE:
        mono = torchaudio.functional.resample(mono, sample_rate, ECAPA_SAMPLE_RATE)
    if mono.numel() == 0:
        raise ValueError('No se puede extraer embedding de un audio vacío.')
    return mono


def speaker_embedding(audio_16k):
    batch = audio_16k.unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        embedding = ecapa.encode_batch(batch).squeeze()
    return F.normalize(embedding.reshape(1, -1), dim=1)


real_second_audio, real_second_sr = load_audio(SECOND_SPEAKER_PATH)
reference_embedding = speaker_embedding(prepare_for_ecapa(reference_clip, reference_sr))
second_embedding = speaker_embedding(prepare_for_ecapa(real_second_audio, real_second_sr))
print('Dimensión del embedding:', reference_embedding.shape[-1])

In [ ]:
similarity_rows = []
for item in generated:
    generated_embedding = speaker_embedding(
        prepare_for_ecapa(item['audio'], item['sample_rate'])
    )
    similarity_rows.append({
        'archivo': item['path'].name,
        'texto': item['text'],
        'coseno vs referencia': F.cosine_similarity(
            generated_embedding, reference_embedding
        ).item(),
        'coseno vs segunda grabación real': F.cosine_similarity(
            generated_embedding, second_embedding
        ).item(),
    })

similarity_table = pd.DataFrame(similarity_rows)
similarity_table

Una cosine similarity mayor indica embeddings más semejantes, pero **no es una probabilidad de identidad ni define un umbral universal**. Interpreta los scores junto con la escucha: no verifican que el texto sea correcto ni que la voz resulte natural.

## Cierre

1. ¿Qué información conserva cada etapa y qué necesita la siguiente?
2. ¿Por qué una transcripción correcta y un embedding parecido responden a preguntas distintas?
3. ¿Qué cambiaría al usar solo el embedding de Qwen en vez de audio y transcripción?
4. ¿Qué fallos aparecen al conectar el sistema completo que no se veían al probar cada componente por separado? Explica al menos un fallo y cómo lo trataste.